# spinangle — gated spherical nGPT-JEPA vs. official LeWM (Colab GPU)

Baseline = **official LeWM, unchanged** (`lucas-maes/le-wm`). The big cell below clones the harness, installs the LeWM stack, reproduces LeWM (Phase 1), then trains/evals the nGPT-JEPA variants on its own benchmark + planner.

**Setup:** Runtime → GPU. The repo is private — either make it public, or set a `GITHUB_TOKEN` Colab Secret / paste a PAT into `GH_TOKEN`. Start with `BENCH='tworoom'` + `EPOCHS=5` to validate; then scale to 100 and add variants.

## ▶️ The big cell (edit config, run)

In [ ]:
#@title 🌀 spinangle: gated spherical nGPT-JEPA vs official LeWM — one-shot runner
import os, subprocess, glob

# ----------------------------- config -----------------------------
BENCH    = "tworoom"   # tworoom (3.4G, lightest) | pusht (13G) | reacher (24G) | cube (46G)
EPOCHS   = 5           # 5 = quick validation; 100 = matched-compute comparison
VARIANTS = ["official_lewm", "gated_spherical"]   # + lewm_nosigreg, simple_spherical, fullish_residual,
#            gated_spherical_projector_sigreg, gated_spherical_memory, gated_spherical_ssm, ngpt_lr ...
GET_DATA = True
BRANCH   = "claude/upbeat-babbage-kbmgsr"
# spinangle is a PRIVATE repo. Pick ONE: (A) make it public and leave GH_TOKEN="",
# (B) paste a GitHub PAT below, or (C) add a Colab Secret named GITHUB_TOKEN (key icon, sidebar).
GH_TOKEN = ""
# ------------------------------------------------------------------

try:
    from google.colab import userdata
    GH_TOKEN = GH_TOKEN or (userdata.get("GITHUB_TOKEN") or "")
except Exception:
    pass

DATACFG = {"tworoom": "tworoom", "pusht": "pusht", "reacher": "dmc", "cube": "ogb"}[BENCH]
os.environ["STABLEWM_HOME"] = "/content/stable-wm"
os.environ["MUJOCO_GL"] = "egl"

def sh(cmd, check=False):
    print(f"\n\033[1;36m$ {cmd}\033[0m", flush=True)
    return subprocess.run(cmd, shell=True, check=check).returncode

sh("nvidia-smi -L || echo '⚠️  NO GPU — Runtime > Change runtime type > GPU'")

# --- clone the private repo (token-aware; fails loudly with instructions) -----
if not os.path.isdir("/content/spinangle/.git"):
    auth = f"{GH_TOKEN}@" if GH_TOKEN else ""
    r = subprocess.run(
        f"git clone -b {BRANCH} https://{auth}github.com/turtlenottortoise/spinangle.git /content/spinangle",
        shell=True, capture_output=True, text=True)
    print(r.stdout); print(r.stderr)
    if not os.path.isdir("/content/spinangle/.git"):
        raise SystemExit(
            "\n❌ Clone failed — 'turtlenottortoise/spinangle' is PRIVATE. Do ONE of:\n"
            "   (A) GitHub > repo > Settings > Change visibility > Public, then rerun; or\n"
            "   (B) make a PAT (github.com/settings/tokens, repo-read scope) and paste it into\n"
            "       GH_TOKEN above, or add it as a Colab Secret named GITHUB_TOKEN, then rerun.")
else:
    sh("cd /content/spinangle && git pull")
os.chdir("/content/spinangle")

# install + CPU smoke test (validates every variant; no data/GPU needed) --------
sh("pip -q install 'stable-worldmodel[train,env]' matplotlib einops huggingface_hub")
sh("python smoke_test.py && python metrics.py", check=True)

# official data + pretrained checkpoint; PHASE 1 reproduce (the gate) -----------
sh(f"python scripts/download_assets.py --benchmark {BENCH} --ckpt" + (" --data" if GET_DATA else ""))
sh(f"python eval.py --config-name={BENCH}.yaml policy={BENCH}/lewm")

# PHASES 2-3 train + eval variants (same eval.py + planner; only +experiment) ---
CKPT = f"{os.environ['STABLEWM_HOME']}/checkpoints/{BENCH}"
for v in VARIANTS:
    sh(f"python train.py +experiment={v} data={DATACFG} "
       f"output_model_name={BENCH}/{v} trainer.max_epochs={EPOCHS} wandb.enabled=false")
    pts = sorted(glob.glob(f"{CKPT}/{v}/weights_epoch_*.pt"), key=os.path.getmtime)
    for old in pts[:-1]:               # keep newest ckpt so load_pretrained finds one .pt
        os.remove(old)
    sh(f"python eval.py --config-name={BENCH}.yaml policy={BENCH}/{v} "
       f"|| echo '⚠️ trained-ckpt eval path — see RUN_MATRIX.md'")
    sph = "" if v in ("official_lewm", "lewm_nosigreg") else "--spherical"
    sh(f"python scripts/eval_latent_metrics.py --policy {BENCH}/{v} --data {DATACFG} "
       f"--benchmark {BENCH} --variant {v} {sph} --horizon 20 --num_batches 16 || true")

# plots ------------------------------------------------------------------------
sh("python scripts/make_plots.py")
from IPython.display import Image, display
for p in ["success_vs_steps", "rollout_error_vs_horizon", "retrieval_vs_steps",
          "rank_clumping", "planning_budget_curve"]:
    fp = f"/content/spinangle/plots/{p}.png"
    if os.path.exists(fp):
        display(Image(fp))
print("\n✅ DONE — results in results/all_runs.csv, plots in plots/. Fill report.md.")


## Phase 7 — νGPT scaling (optional, run after the loop above)

In [ ]:
BENCH, DATACFG, EPOCHS = 'tworoom', 'tworoom', 100
for v in ['gated_spherical', 'ngpt_lr', 'ngpt_lr_groups']:
    !python train.py +experiment={v} data={DATACFG} output_model_name={BENCH}/{v} \
        trainer.max_epochs={EPOCHS} wandb.enabled=false
    !python eval.py --config-name={BENCH}.yaml policy={BENCH}/{v} || true


## Persist results back to the branch (optional)

In [ ]:
!cd /content/spinangle && git add results/all_runs.csv plots/*.png && \
  git -c user.email=colab@local -c user.name=colab commit -m 'colab: results' && \
  git push || echo 'configure git auth (token) to push'
